In [27]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import re
import math
import numpy as np
import pandas as pd

import nltk
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download NLTK stopwords if necessary.
try:
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
    STOPWORDS = set(stopwords.words("english"))
except Exception:
    # Fallback so the notebook can still run if NLTK data is unavailable.
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    STOPWORDS = set(ENGLISH_STOP_WORDS)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

print("Libraries imported successfully.")
print("Number of stopwords:", len(STOPWORDS))


Libraries imported successfully.
Number of stopwords: 198


In [28]:
def preprocess_text(text):
    """Lowercase, tokenize alphabetic words, and remove English stopwords."""
    text = str(text).lower()
    tokens = re.findall(r"[a-z]+", text)
    tokens = [token for token in tokens if token not in STOPWORDS]
    return tokens


def preprocess_corpus(corpus):
    return [preprocess_text(document) for document in corpus]


def show_corpus(corpus, processed):
    for i, (original, tokens) in enumerate(zip(corpus, processed), 1):
        print(f"D{i} Original : {original}")
        print(f"D{i} Processed: {tokens}")
        print()


# Q1. Implement Term Frequency and TF-IDF Representation




In [29]:
# ------------------------------------------------------------
# Q1(a) Small manually created corpus: 5 documents
# ------------------------------------------------------------

q1_corpus = [
    "Artificial intelligence helps computers learn from data. Machine learning is a part of artificial intelligence.",
    "Machine learning algorithms learn useful patterns from data. Data science uses machine learning for prediction.",
    "Natural language processing helps computers understand human language. Language models learn from text data.",
    "Deep learning uses neural networks to learn complex patterns. Neural networks are widely used in artificial intelligence.",
    "Data science combines statistics, programming, and machine learning to solve practical problems."
]

q1_processed = preprocess_corpus(q1_corpus)

print("Q1 Corpus and Preprocessed Corpus")
print("=" * 70)
show_corpus(q1_corpus, q1_processed)


Q1 Corpus and Preprocessed Corpus
D1 Original : Artificial intelligence helps computers learn from data. Machine learning is a part of artificial intelligence.
D1 Processed: ['artificial', 'intelligence', 'helps', 'computers', 'learn', 'data', 'machine', 'learning', 'part', 'artificial', 'intelligence']

D2 Original : Machine learning algorithms learn useful patterns from data. Data science uses machine learning for prediction.
D2 Processed: ['machine', 'learning', 'algorithms', 'learn', 'useful', 'patterns', 'data', 'data', 'science', 'uses', 'machine', 'learning', 'prediction']

D3 Original : Natural language processing helps computers understand human language. Language models learn from text data.
D3 Processed: ['natural', 'language', 'processing', 'helps', 'computers', 'understand', 'human', 'language', 'language', 'models', 'learn', 'text', 'data']

D4 Original : Deep learning uses neural networks to learn complex patterns. Neural networks are widely used in artificial intelligen

In [30]:
# ------------------------------------------------------------
# Q1(c) Construct vocabulary
# ------------------------------------------------------------

q1_vocabulary = sorted(set(
    token
    for document in q1_processed
    for token in document
))

print("Vocabulary size:", len(q1_vocabulary))
print("Vocabulary:")
print(q1_vocabulary)


Vocabulary size: 34
Vocabulary:
['algorithms', 'artificial', 'combines', 'complex', 'computers', 'data', 'deep', 'helps', 'human', 'intelligence', 'language', 'learn', 'learning', 'machine', 'models', 'natural', 'networks', 'neural', 'part', 'patterns', 'practical', 'prediction', 'problems', 'processing', 'programming', 'science', 'solve', 'statistics', 'text', 'understand', 'used', 'useful', 'uses', 'widely']


In [31]:
# ------------------------------------------------------------
# Q1(d) Calculate Term Frequency manually
# ------------------------------------------------------------

tf_matrix = pd.DataFrame(
    0.0,
    index=[f"D{i}" for i in range(1, len(q1_processed) + 1)],
    columns=q1_vocabulary
)

for doc_index, tokens in enumerate(q1_processed):
    counts = Counter(tokens)
    total_terms = len(tokens)

    for term in q1_vocabulary:
        tf_matrix.loc[f"D{doc_index + 1}", term] = counts[term] / total_terms

print("TF Matrix:")
display(tf_matrix.round(4))
print("TF matrix dimensions:", tf_matrix.shape)


TF Matrix:


,algorithms,artificial,combines,complex,computers,data,deep,helps,human,intelligence,language,learn,learning,machine,models,natural,networks,neural,part,patterns,practical,prediction,problems,processing,programming,science,solve,statistics,text,understand,used,useful,uses,widely
D1,0.0000,0.1818,0.0,0.0000,0.0909,0.0909,0.0000,0.0909,0.0000,0.1818,0.0000,0.0909,0.0909,0.0909,0.0000,0.0000,0.0000,0.0000,0.0909,0.0000,0.0,0.0000,0.0,0.0000,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
D2,0.0769,0.0000,0.0,0.0000,0.0000,0.1538,0.0000,0.0000,0.0000,0.0000,0.0000,0.0769,0.1538,0.1538,0.0000,0.0000,0.0000,0.0000,0.0000,0.0769,0.0,0.0769,0.0,0.0000,0.0,0.0769,0.0,0.0,0.0000,0.0000,0.0000,0.0769,0.0769,0.0000
D3,0.0000,0.0000,0.0,0.0000,0.0769,0.0769,0.0000,0.0769,0.0769,0.0000,0.2308,0.0769,0.0000,0.0000,0.0769,0.0769,0.0000,0.0000,0.0000,0.0000,0.0,0.0000,0.0,0.0769,0.0,0.0000,0.0,0.0,0.0769,0.0769,0.0000,0.0000,0.0000,0.0000
D4,0.0000,0.0714,0.0,0.0714,0.0000,0.0000,0.0714,0.0000,0.0000,0.0714,0.0000,0.0714,0.0714,0.0000,0.0000,0.0000,0.1429,0.1429,0.0000,0.0714,0.0,0.0000,0.0,0.0000,0.0,0.0000,0.0,0.0,0.0000,0.0000,0.0714,0.0000,0.0714,0.0714
D5,0.0000,0.0000,0.1,0.0000,0.0000,0.1000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1000,0.1000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1,0.0000,0.1,0.0000,0.1,0.1000,0.1,0.1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


TF matrix dimensions: (5, 34)


In [32]:
# ------------------------------------------------------------
# Q1(e) Calculate IDF manually
# ------------------------------------------------------------

N = len(q1_processed)

idf_values = {}

for term in q1_vocabulary:
    document_frequency = sum(
        term in set(tokens)
        for tokens in q1_processed
    )
    idf_values[term] = math.log(N / document_frequency)

idf_series = pd.Series(idf_values, name="IDF").sort_index()

print("IDF Values:")
display(idf_series.round(4).to_frame())


IDF Values:


,IDF
algorithms,1.6094
artificial,0.9163
combines,1.6094
complex,1.6094
computers,0.9163
data,0.2231
deep,1.6094
helps,0.9163
human,1.6094
intelligence,0.9163


In [33]:
# ------------------------------------------------------------
# Q1(f) Calculate TF-IDF manually
# ------------------------------------------------------------

q1_tfidf_matrix = tf_matrix.copy()

for term in q1_vocabulary:
    q1_tfidf_matrix[term] = (
        tf_matrix[term] * idf_series[term]
    )

print("TF-IDF Matrix:")
display(q1_tfidf_matrix.round(4))
print("TF-IDF matrix dimensions:", q1_tfidf_matrix.shape)


TF-IDF Matrix:


,algorithms,artificial,combines,complex,computers,data,deep,helps,human,intelligence,language,learn,learning,machine,models,natural,networks,neural,part,patterns,practical,prediction,problems,processing,programming,science,solve,statistics,text,understand,used,useful,uses,widely
D1,0.0000,0.1666,0.0000,0.000,0.0833,0.0203,0.000,0.0833,0.0000,0.1666,0.0000,0.0203,0.0203,0.0464,0.0000,0.0000,0.0000,0.0000,0.1463,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000
D2,0.1238,0.0000,0.0000,0.000,0.0000,0.0343,0.000,0.0000,0.0000,0.0000,0.0000,0.0172,0.0343,0.0786,0.0000,0.0000,0.0000,0.0000,0.0000,0.0705,0.0000,0.1238,0.0000,0.0000,0.0000,0.0705,0.0000,0.0000,0.0000,0.0000,0.000,0.1238,0.0705,0.000
D3,0.0000,0.0000,0.0000,0.000,0.0705,0.0172,0.000,0.0705,0.1238,0.0000,0.3714,0.0172,0.0000,0.0000,0.1238,0.1238,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1238,0.0000,0.0000,0.0000,0.0000,0.1238,0.1238,0.000,0.0000,0.0000,0.000
D4,0.0000,0.0654,0.0000,0.115,0.0000,0.0000,0.115,0.0000,0.0000,0.0654,0.0000,0.0159,0.0159,0.0000,0.0000,0.0000,0.2299,0.2299,0.0000,0.0654,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.115,0.0000,0.0654,0.115
D5,0.0000,0.0000,0.1609,0.000,0.0000,0.0223,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0223,0.0511,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1609,0.0000,0.1609,0.0000,0.1609,0.0916,0.1609,0.1609,0.0000,0.0000,0.000,0.0000,0.0000,0.000


TF-IDF matrix dimensions: (5, 34)


In [34]:
# ------------------------------------------------------------
# Q1(g,h) Identify important terms in every document
# ------------------------------------------------------------

TOP_K = 5

print("Most important terms in each document:")
print("=" * 70)

q1_important_terms = {}

for document in q1_tfidf_matrix.index:
    top_terms = (
        q1_tfidf_matrix.loc[document]
        .sort_values(ascending=False)
        .head(TOP_K)
    )

    q1_important_terms[document] = top_terms

    print(f"\n{document} - Top {TOP_K} terms")
    for term, score in top_terms.items():
        print(f"{term:20s} -> {score:.4f}")


Most important terms in each document:

D1 - Top 5 terms
artificial           -> 0.1666
intelligence         -> 0.1666
part                 -> 0.1463
helps                -> 0.0833
computers            -> 0.0833

D2 - Top 5 terms
algorithms           -> 0.1238
useful               -> 0.1238
prediction           -> 0.1238
machine              -> 0.0786
patterns             -> 0.0705

D3 - Top 5 terms
language             -> 0.3714
human                -> 0.1238
understand           -> 0.1238
text                 -> 0.1238
models               -> 0.1238

D4 - Top 5 terms
networks             -> 0.2299
neural               -> 0.2299
widely               -> 0.1150
used                 -> 0.1150
complex              -> 0.1150

D5 - Top 5 terms
programming          -> 0.1609
combines             -> 0.1609
solve                -> 0.1609
practical            -> 0.1609
problems             -> 0.1609


# Q2. Implement the Vector Space Model and Document Similarity




In [35]:
# ------------------------------------------------------------
# Q2(a) 10 short documents from a common domain
# ------------------------------------------------------------

q2_documents = [
    "Artificial intelligence enables computers to perform tasks that normally require human intelligence.",
    "Machine learning allows computers to learn patterns from data and make predictions.",
    "Deep learning uses neural networks with many layers to learn complex representations from large datasets.",
    "Natural language processing enables computers to analyze and understand human language.",
    "Computer vision allows machines to understand images and videos using image processing and neural networks.",
    "Cloud computing provides scalable computing storage and software services through the internet.",
    "Cyber security protects computer systems networks and data from unauthorized access and attacks.",
    "Data science combines statistics programming and machine learning to discover useful information from data.",
    "Artificial intelligence and machine learning are widely used in healthcare for prediction and diagnosis.",
    "Neural networks are used in computer vision and deep learning to recognize patterns in images."
]

q2_processed = preprocess_corpus(q2_documents)

print("Q2 Original and Preprocessed Documents")
print("=" * 80)
show_corpus(q2_documents, q2_processed)


Q2 Original and Preprocessed Documents
D1 Original : Artificial intelligence enables computers to perform tasks that normally require human intelligence.
D1 Processed: ['artificial', 'intelligence', 'enables', 'computers', 'perform', 'tasks', 'normally', 'require', 'human', 'intelligence']

D2 Original : Machine learning allows computers to learn patterns from data and make predictions.
D2 Processed: ['machine', 'learning', 'allows', 'computers', 'learn', 'patterns', 'data', 'make', 'predictions']

D3 Original : Deep learning uses neural networks with many layers to learn complex representations from large datasets.
D3 Processed: ['deep', 'learning', 'uses', 'neural', 'networks', 'many', 'layers', 'learn', 'complex', 'representations', 'large', 'datasets']

D4 Original : Natural language processing enables computers to analyze and understand human language.
D4 Processed: ['natural', 'language', 'processing', 'enables', 'computers', 'analyze', 'understand', 'human', 'language']

D5 Orig

In [36]:
# ------------------------------------------------------------
# Q2(b,c) Generate TF-IDF document vectors
# ------------------------------------------------------------

# TfidfVectorizer performs tokenization, lowercasing,
# stopword removal and TF-IDF representation.
q2_vectorizer = TfidfVectorizer(stop_words="english")

q2_matrix = q2_vectorizer.fit_transform(q2_documents)
q2_features = q2_vectorizer.get_feature_names_out()

print("Vocabulary size:", len(q2_features))
print("Document-term matrix dimensions:", q2_matrix.shape)

q2_dense = pd.DataFrame(
    q2_matrix.toarray(),
    index=[f"D{i}" for i in range(1, 11)],
    columns=q2_features
)

print("\nDocument-Term Matrix:")
display(q2_dense.round(4))


Vocabulary size: 66
Document-term matrix dimensions: (10, 66)

Document-Term Matrix:


,access,allows,analyze,artificial,attacks,cloud,combines,complex,computer,computers,computing,cyber,data,datasets,deep,diagnosis,discover,enables,healthcare,human,image,images,information,intelligence,internet,language,large,layers,learn,learning,machine,machines,make,natural,networks,neural,normally,patterns,perform,prediction,predictions,processing,programming,protects,provides,recognize,representations,require,scalable,science,security,services,software,statistics,storage,systems,tasks,unauthorized,understand,used,useful,uses,using,videos,vision,widely
D1,0.0000,0.0000,0.0000,0.2742,0.0000,0.0000,0.0000,0.0000,0.0000,0.2399,0.000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.2742,0.000,0.2742,0.0000,0.0000,0.0000,0.5484,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3226,0.0000,0.3226,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3226,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3226,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000
D2,0.0000,0.3420,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2992,0.000,0.0000,0.2992,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3420,0.2389,0.2992,0.0000,0.4023,0.0000,0.0000,0.0000,0.0000,0.3420,0.0000,0.000,0.4023,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000
D3,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3373,0.0000,0.0000,0.000,0.0000,0.0000,0.3373,0.2868,0.000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3373,0.3373,0.2868,0.2003,0.0000,0.0000,0.0000,0.0000,0.2230,0.2509,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3373,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3373,0.0000,0.0000,0.0000,0.000
D4,0.0000,0.0000,0.3254,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2420,0.000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.2766,0.000,0.2766,0.0000,0.0000,0.0000,0.0000,0.0000,0.6508,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3254,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.2766,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2766,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000
D5,0.0000,0.2809,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2458,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,0.3305,0.2809,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3305,0.0000,0.0000,0.2185,0.2458,0.0000,0.0000,0.0000,0.000,0.0000,0.2809,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2809,0.0000,0.0000,0.0000,0.3305,0.3305,0.2809,0.000
D6,0.0000,0.0000,0.0000,0.0000,0.0000,0.3015,0.0000,0.0000,0.0000,0.0000,0.603,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3015,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.3015,0.0000,0.0000,0.0000,0.3015,0.0000,0.0000,0.3015,0.3015,0.0000,0.3015,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000
D7,0.3421,0.0000,0.0000,0.0000,0.3421,0.0000,0.0000,0.0000,0.2544,0.0000,0.000,0.3421,0.2544,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2262,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.3421,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3421,0.0000,0.0000,0.0000,0.0000,0.3421,0.0000,0.3421,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000
D8,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3144,0.0000,0.0000,0.0000,0.000,0.0000,0.4676,0.0000,0.0000,0.000,0.3144,0.0000,0.000,0.0000,0.0000,0.0000,0.3144,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1867,0.2338,0

In [37]:
# ------------------------------------------------------------
# Q2(c,d) Display vector representation and dimensions
# ------------------------------------------------------------

print("Vector representation of each document:")
print("=" * 80)

for i in range(q2_matrix.shape[0]):
    vector = q2_matrix[i].toarray().flatten()
    print(
        f"D{i+1}: vector dimension = {len(vector)}, "
        f"first 10 values = {np.round(vector[:10], 4)}"
    )

print("\nComplete document-term matrix shape:", q2_matrix.shape)
print("Therefore each document is represented by a", q2_matrix.shape[1], "dimensional vector.")


Vector representation of each document:
D1: vector dimension = 66, first 10 values = [0.     0.     0.     0.2742 0.     0.     0.     0.     0.     0.2399]
D2: vector dimension = 66, first 10 values = [0.     0.342  0.     0.     0.     0.     0.     0.     0.     0.2992]
D3: vector dimension = 66, first 10 values = [0.     0.     0.     0.     0.     0.     0.     0.3373 0.     0.    ]
D4: vector dimension = 66, first 10 values = [0.     0.     0.3254 0.     0.     0.     0.     0.     0.     0.242 ]
D5: vector dimension = 66, first 10 values = [0.     0.2809 0.     0.     0.     0.     0.     0.     0.2458 0.    ]
D6: vector dimension = 66, first 10 values = [0.     0.     0.     0.     0.     0.3015 0.     0.     0.     0.    ]
D7: vector dimension = 66, first 10 values = [0.3421 0.     0.     0.     0.3421 0.     0.     0.     0.2544 0.    ]
D8: vector dimension = 66, first 10 values = [0.     0.     0.     0.     0.     0.     0.3144 0.     0.     0.    ]
D9: vector dimension = 6

In [38]:
# ------------------------------------------------------------
# Q2(e) Cosine similarity for at least 3 document pairs
# ------------------------------------------------------------

q2_pairs = [
    (1, 9),
    (3, 10),
    (5, 10),
    (2, 8),
    (4, 1)
]

q2_pair_results = []

print("Cosine Similarity Scores")
print("=" * 50)

for a, b in q2_pairs:
    score = cosine_similarity(
        q2_matrix[a - 1],
        q2_matrix[b - 1]
    )[0, 0]

    q2_pair_results.append([f"D{a}", f"D{b}", score])
    print(f"D{a} vs D{b}: {score:.4f}")

q2_pair_df = pd.DataFrame(
    q2_pair_results,
    columns=["Document 1", "Document 2", "Cosine Similarity"]
)

display(q2_pair_df.round(4))


Cosine Similarity Scores
D1 vs D9: 0.2629
D3 vs D10: 0.2731
D5 vs D10: 0.3871
D2 vs D8: 0.2544
D4 vs D1: 0.2098


,Document 1,Document 2,Cosine Similarity
0,D1,D9,0.2629
1,D3,D10,0.2731
2,D5,D10,0.3871
3,D2,D8,0.2544
4,D4,D1,0.2098


In [39]:
# ------------------------------------------------------------
# Q2(f,g) Full similarity matrix and most similar pair
# ------------------------------------------------------------

q2_similarity = cosine_similarity(q2_matrix)

q2_similarity_df = pd.DataFrame(
    q2_similarity,
    index=[f"D{i}" for i in range(1, 11)],
    columns=[f"D{i}" for i in range(1, 11)]
)

print("Complete Document Similarity Matrix:")
display(q2_similarity_df.round(4))

# Ignore diagonal because each document has similarity 1 with itself.
sim_copy = q2_similarity.copy()
np.fill_diagonal(sim_copy, -1)

max_position = np.unravel_index(
    np.argmax(sim_copy),
    sim_copy.shape
)

d1 = max_position[0] + 1
d2 = max_position[1] + 1
max_score = sim_copy[max_position]

print(f"Most similar pair in this corpus: D{d1} and D{d2}")
print(f"Cosine similarity: {max_score:.4f}")

print("\nObservation:")
print("Documents with greater lexical overlap in important TF-IDF terms tend to have higher cosine similarity.")


Complete Document Similarity Matrix:


,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
D1,1.0000,0.0718,0.0000,0.2098,0.0000,0.0,0.0000,0.0000,0.2629,0.0000
D2,0.0718,1.0000,0.1459,0.0724,0.0961,0.0,0.0761,0.2544,0.1370,0.1695
D3,0.0000,0.1459,1.0000,0.0000,0.1104,0.0,0.0505,0.0374,0.0447,0.2731
D4,0.2098,0.0724,0.0000,1.0000,0.1554,0.0,0.0000,0.0000,0.0000,0.0000
D5,0.0000,0.0961,0.1104,0.1554,1.0000,0.0,0.1120,0.0000,0.0000,0.3871
D6,0.0000,0.0000,0.0000,0.0000,0.0000,1.0,0.0000,0.0000,0.0000,0.0000
D7,0.0000,0.0761,0.0505,0.0000,0.1120,0.0,1.0000,0.1190,0.0000,0.1328
D8,0.0000,0.2544,0.0374,0.0000,0.0000,0.0,0.1190,1.0000,0.1071,0.0434
D9,0.2629,0.1370,0.0447,0.0000,0.0000,0.0,0.0000,0.1071,1.0000,0.1585
D10,0.0000,0.1695,0.2731,0.0000,0.3871,0.0,0.1328,0.0434,0.1585,1.0000


Most similar pair in this corpus: D5 and D10
Cosine similarity: 0.3871

Observation:
Documents with greater lexical overlap in important TF-IDF terms tend to have higher cosine similarity.


# Q3. Study Distributional Representation and Word Representation



In [40]:
# ------------------------------------------------------------
# Q3(a) Corpus containing 40 sentences
# ------------------------------------------------------------

q3_sentences = [
    "Machine learning learns patterns from data.",
    "Machine learning models make predictions from data.",
    "Deep learning uses neural networks for complex tasks.",
    "Neural networks learn representations from training data.",
    "Artificial intelligence includes machine learning methods.",
    "Artificial intelligence can solve complex problems.",
    "Data science uses statistics and machine learning.",
    "Data science extracts useful information from data.",
    "Natural language processing works with human language.",
    "Natural language processing analyzes text data.",
    "Computer vision analyzes images and videos.",
    "Computer vision uses neural networks for recognition.",
    "Python is widely used for machine learning.",
    "Python provides libraries for data science.",
    "TensorFlow supports deep learning models.",
    "PyTorch is used to build neural networks.",
    "Models require training data to learn patterns.",
    "Training data is used to train machine learning models.",
    "Testing data evaluates machine learning models.",
    "Large datasets can improve model performance.",
    "Feature engineering creates useful features from data.",
    "Features help models identify important patterns.",
    "Classification predicts categories from input data.",
    "Regression predicts numerical values from data.",
    "Clustering groups similar data points.",
    "Algorithms process data to produce predictions.",
    "A model learns relationships between features and targets.",
    "Optimization improves the parameters of a model.",
    "Gradient descent is used to optimize neural networks.",
    "Neural networks contain layers of connected neurons.",
    "Deep networks contain multiple hidden layers.",
    "Computer vision models recognize objects in images.",
    "Language models learn patterns in text.",
    "Text data contains words and sentences.",
    "Words occurring in similar contexts can have related meanings.",
    "Context provides useful information about a word.",
    "Word representations can be created from co-occurrence information.",
    "Vector representations allow mathematical comparison of words.",
    "Cosine similarity compares the direction of word vectors.",
    "Similar vectors indicate similar distributional contexts."
]

assert len(q3_sentences) == 40

q3_processed = preprocess_corpus(q3_sentences)

print("Number of sentences:", len(q3_processed))
print("\nFirst 10 preprocessed sentences:")
for i, tokens in enumerate(q3_processed[:10], 1):
    print(f"S{i}: {tokens}")


Number of sentences: 40

First 10 preprocessed sentences:
S1: ['machine', 'learning', 'learns', 'patterns', 'data']
S2: ['machine', 'learning', 'models', 'make', 'predictions', 'data']
S3: ['deep', 'learning', 'uses', 'neural', 'networks', 'complex', 'tasks']
S4: ['neural', 'networks', 'learn', 'representations', 'training', 'data']
S5: ['artificial', 'intelligence', 'includes', 'machine', 'learning', 'methods']
S6: ['artificial', 'intelligence', 'solve', 'complex', 'problems']
S7: ['data', 'science', 'uses', 'statistics', 'machine', 'learning']
S8: ['data', 'science', 'extracts', 'useful', 'information', 'data']
S9: ['natural', 'language', 'processing', 'works', 'human', 'language']
S10: ['natural', 'language', 'processing', 'analyzes', 'text', 'data']


In [41]:
# ------------------------------------------------------------
# Q3(b) Construct complete vocabulary
# ------------------------------------------------------------

q3_vocabulary = sorted(set(
    token
    for sentence in q3_processed
    for token in sentence
))

print("Complete Q3 Vocabulary:")
print(q3_vocabulary)
print("\nVocabulary size:", len(q3_vocabulary))


Complete Q3 Vocabulary:
['algorithms', 'allow', 'analyzes', 'artificial', 'build', 'categories', 'classification', 'clustering', 'co', 'compares', 'comparison', 'complex', 'computer', 'connected', 'contain', 'contains', 'context', 'contexts', 'cosine', 'created', 'creates', 'data', 'datasets', 'deep', 'descent', 'direction', 'distributional', 'engineering', 'evaluates', 'extracts', 'feature', 'features', 'gradient', 'groups', 'help', 'hidden', 'human', 'identify', 'images', 'important', 'improve', 'improves', 'includes', 'indicate', 'information', 'input', 'intelligence', 'language', 'large', 'layers', 'learn', 'learning', 'learns', 'libraries', 'machine', 'make', 'mathematical', 'meanings', 'methods', 'model', 'models', 'multiple', 'natural', 'networks', 'neural', 'neurons', 'numerical', 'objects', 'occurrence', 'occurring', 'optimization', 'optimize', 'parameters', 'patterns', 'performance', 'points', 'predictions', 'predicts', 'problems', 'process', 'processing', 'produce', 'provide

In [42]:
# ------------------------------------------------------------
# Q3(c,d) Select at least 10 frequently occurring words
# ------------------------------------------------------------

q3_word_counts = Counter(
    token
    for sentence in q3_processed
    for token in sentence
)

frequency_df = pd.DataFrame(
    q3_word_counts.most_common(),
    columns=["Word", "Frequency"]
)

print("Word Frequencies:")
display(frequency_df)

# Select 15 frequent words, which is greater than the required 10.
selected_words = [
    word
    for word, count in q3_word_counts.most_common(15)
]

print("Selected frequent words (15):")
print(selected_words)


Word Frequencies:


,Word,Frequency
0,data,17
1,learning,9
2,models,8
3,machine,7
4,networks,7
...,...,...
113,similarity,1
114,compares,1
115,direction,1
116,indicate,1


Selected frequent words (15):
['data', 'learning', 'models', 'machine', 'networks', 'neural', 'patterns', 'language', 'used', 'similar', 'deep', 'uses', 'learn', 'representations', 'training']


In [43]:
# ------------------------------------------------------------
# Q3(c,e) Build a word-context co-occurrence matrix
# ------------------------------------------------------------

# Use the complete vocabulary as context dimensions.
context_words = q3_vocabulary

window_size = 2

cooccurrence = pd.DataFrame(
    0.0,
    index=selected_words,
    columns=context_words
)

for tokens in q3_processed:
    for i, word in enumerate(tokens):

        if word not in selected_words:
            continue

        start = max(0, i - window_size)
        end = min(len(tokens), i + window_size + 1)

        for j in range(start, end):
            if i == j:
                continue

            context = tokens[j]

            if context in context_words:
                cooccurrence.loc[word, context] += 1

print("Co-occurrence matrix dimensions:", cooccurrence.shape)
print(
    "Rows = selected words;",
    "columns = complete vocabulary context words."
)

display(cooccurrence)


Co-occurrence matrix dimensions: (15, 118)
Rows = selected words; columns = complete vocabulary context words.


,algorithms,allow,analyzes,artificial,build,categories,classification,clustering,co,compares,comparison,complex,computer,connected,contain,contains,context,contexts,cosine,created,creates,data,datasets,deep,descent,direction,distributional,engineering,evaluates,extracts,feature,features,gradient,groups,help,hidden,human,identify,images,important,improve,improves,includes,indicate,information,input,intelligence,language,large,layers,...,occurrence,occurring,optimization,optimize,parameters,patterns,performance,points,predictions,predicts,problems,process,processing,produce,provides,python,pytorch,recognition,recognize,regression,related,relationships,representations,require,science,sentences,similar,similarity,solve,statistics,supports,targets,tasks,tensorflow,testing,text,train,training,used,useful,uses,values,vector,vectors,videos,vision,widely,word,words,works
data,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,1.0,2.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,1.0,3.0,1.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
learning,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
models,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
machine,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
networks,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
neural,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
patterns,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
language,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.

In [44]:
# ------------------------------------------------------------
# Q3(e,f) Word vectors and cosine similarity
# ------------------------------------------------------------

# Each row of the co-occurrence matrix is a word vector.
word_vectors = cooccurrence.values

print("Word-vector matrix dimensions:", word_vectors.shape)

def word_similarity(word1, word2):
    i = selected_words.index(word1)
    j = selected_words.index(word2)

    return cosine_similarity(
        word_vectors[i].reshape(1, -1),
        word_vectors[j].reshape(1, -1)
    )[0, 0]

# Pairs are selected from the actual frequent vocabulary.
candidate_pairs = [
    ("machine", "learning"),
    ("data", "model"),
    ("neural", "networks"),
    ("data", "features"),
    ("models", "networks")
]

results = []

print("Selected word-vector similarity scores:")
print("=" * 60)

for word1, word2 in candidate_pairs:
    if word1 in selected_words and word2 in selected_words:
        score = word_similarity(word1, word2)
        results.append([word1, word2, score])
        print(f"{word1:12s} vs {word2:12s}: {score:.4f}")
    else:
        print(f"{word1} vs {word2}: one/both words not selected")

word_similarity_pairs = pd.DataFrame(
    results,
    columns=["Word 1", "Word 2", "Cosine Similarity"]
)

display(word_similarity_pairs.round(4))


Word-vector matrix dimensions: (15, 118)
Selected word-vector similarity scores:
machine      vs learning    : 0.2750
data vs model: one/both words not selected
neural       vs networks    : 0.2076
data vs features: one/both words not selected
models       vs networks    : 0.0425


,Word 1,Word 2,Cosine Similarity
0,machine,learning,0.2750
1,neural,networks,0.2076
2,models,networks,0.0425


In [45]:
# ------------------------------------------------------------
# Q3(g) Similarity matrix and most similar word
# ------------------------------------------------------------

word_similarity_matrix = cosine_similarity(word_vectors)

word_similarity_df = pd.DataFrame(
    word_similarity_matrix,
    index=selected_words,
    columns=selected_words
)

print("Cosine Similarity Matrix for Selected Words:")
display(word_similarity_df.round(4))

print("Most similar other word for each selected word:")
print("=" * 60)

for i, word in enumerate(selected_words):
    scores = word_similarity_matrix[i].copy()
    scores[i] = -1

    j = np.argmax(scores)

    print(
        f"{word:15s} -> "
        f"{selected_words[j]:15s} "
        f"({scores[j]:.4f})"
    )


Cosine Similarity Matrix for Selected Words:


,data,learning,models,machine,networks,neural,patterns,language,used,similar,deep,uses,learn,representations,training
data,1.0000,0.2100,0.2755,0.0891,0.0686,0.0874,0.1684,0.0630,0.1734,0.0845,0.0398,0.1260,0.4229,0.1594,0.1528
learning,0.2100,1.0000,0.4338,0.2750,0.1512,0.0616,0.1485,0.1111,0.4333,0.0000,0.2108,0.3333,0.2008,0.0000,0.1347
models,0.2755,0.4338,1.0000,0.5153,0.0425,0.1299,0.3339,0.0390,0.3941,0.0000,0.3951,0.3904,0.2016,0.0988,0.1136
machine,0.0891,0.2750,0.5153,1.0000,0.0321,0.2125,0.4095,0.0884,0.2704,0.0264,0.6708,0.2652,0.1217,0.0373,0.2287
networks,0.0686,0.1512,0.0425,0.0321,1.0000,0.2076,0.0727,0.0340,0.4371,0.0000,0.1721,0.4423,0.2460,0.0430,0.0990
neural,0.0874,0.0616,0.1299,0.2125,0.2076,1.0000,0.1112,0.0347,0.0954,0.0000,0.4824,0.4854,0.2148,0.3070,0.1345
patterns,0.1684,0.1485,0.3339,0.4095,0.0727,0.1112,1.0000,0.2004,0.1839,0.1195,0.2535,0.2004,0.2760,0.3381,0.7130
language,0.0630,0.1111,0.0390,0.0884,0.0340,0.0347,0.2004,1.0000,0.0000,0.0000,0.0791,0.0000,0.0645,0.0791,0.1819
used,0.1734,0.4333,0.3941,0.2704,0.4371,0.0954,0.1839,0.0000,1.0000,0.0513,0.1451,0.4588,0.2962,0.1451,0.1669
similar,0.0845,0.0000,0.0000,0.0264,0.0000,0.0000,0.1195,0.0000,0.0513,1.0000,0.0000,0.0559,0.0577,0.0707,0.1627


Most similar other word for each selected word:
data            -> learn           (0.4229)
learning        -> models          (0.4338)
models          -> machine         (0.5153)
machine         -> deep            (0.6708)
networks        -> uses            (0.4423)
neural          -> uses            (0.4854)
patterns        -> training        (0.7130)
language        -> patterns        (0.2004)
used            -> uses            (0.4588)
similar         -> training        (0.1627)
deep            -> machine         (0.6708)
uses            -> neural          (0.4854)
learn           -> data            (0.4229)
representations -> training        (0.3835)
training        -> patterns        (0.7130)


# Q4. Generate Sentence Representation and Document-to-Vector Representation



In [46]:
# ------------------------------------------------------------
# Q4(a) 12 sentences belonging to two topics
# ------------------------------------------------------------

q4_sentences = [
    "Machine learning models learn useful patterns from training data.",
    "Deep learning uses neural networks to recognize complex patterns.",
    "Artificial intelligence systems can make predictions from data.",
    "Natural language processing helps computers understand human language.",
    "Computer vision models classify objects in images and videos.",
    "Data science uses statistics and machine learning to analyze data.",

    "Cyber security protects systems from unauthorized access.",
    "Network security prevents attacks on computer networks.",
    "Encryption protects sensitive information from unauthorized users.",
    "Firewalls monitor network traffic and block suspicious connections.",
    "Authentication verifies the identity of users before access.",
    "Security monitoring helps detect malicious network activity."
]

q4_labels = [
    "AI", "AI", "AI", "AI", "AI", "AI",
    "Security", "Security", "Security", "Security", "Security", "Security"
]

assert len(q4_sentences) >= 10

print("Q4 Sentences:")
print("=" * 80)

for i, (sentence, label) in enumerate(
    zip(q4_sentences, q4_labels), 1
):
    print(f"S{i} [{label}]: {sentence}")


Q4 Sentences:
S1 [AI]: Machine learning models learn useful patterns from training data.
S2 [AI]: Deep learning uses neural networks to recognize complex patterns.
S3 [AI]: Artificial intelligence systems can make predictions from data.
S4 [AI]: Natural language processing helps computers understand human language.
S5 [AI]: Computer vision models classify objects in images and videos.
S6 [AI]: Data science uses statistics and machine learning to analyze data.
S7 [Security]: Cyber security protects systems from unauthorized access.
S8 [Security]: Network security prevents attacks on computer networks.
S9 [Security]: Encryption protects sensitive information from unauthorized users.
S10 [Security]: Firewalls monitor network traffic and block suspicious connections.
S11 [Security]: Authentication verifies the identity of users before access.
S12 [Security]: Security monitoring helps detect malicious network activity.


In [47]:
# ------------------------------------------------------------
# Q4(b,c) TF-IDF sentence representation
# ------------------------------------------------------------

q4_vectorizer = TfidfVectorizer(stop_words="english")

q4_matrix = q4_vectorizer.fit_transform(q4_sentences)
q4_feature_names = q4_vectorizer.get_feature_names_out()

print("Sentence-term matrix dimensions:", q4_matrix.shape)
print("Each sentence vector dimension:", q4_matrix.shape[1])

q4_vectors = pd.DataFrame(
    q4_matrix.toarray(),
    index=[f"S{i}" for i in range(1, len(q4_sentences) + 1)],
    columns=q4_feature_names
)

print("\nSentence Vector Matrix:")
display(q4_vectors.round(4))


Sentence-term matrix dimensions: (12, 60)
Each sentence vector dimension: 60

Sentence Vector Matrix:


,access,activity,analyze,artificial,attacks,authentication,block,classify,complex,computer,computers,connections,cyber,data,deep,detect,encryption,firewalls,helps,human,identity,images,information,intelligence,language,learn,learning,machine,make,malicious,models,monitor,monitoring,natural,network,networks,neural,objects,patterns,predictions,prevents,processing,protects,recognize,science,security,sensitive,statistics,suspicious,systems,traffic,training,unauthorized,understand,useful,users,uses,verifies,videos,vision
S1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00,0.000,0.0000,0.0000,0.0000,0.00,0.0000,0.3007,0.0000,0.0000,0.000,0.00,0.0000,0.0000,0.0000,0.000,0.000,0.0000,0.0000,0.3964,0.3007,0.3404,0.0000,0.0000,0.3404,0.00,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.3404,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.00,0.0000,0.00,0.3964,0.0000,0.0000,0.3964,0.0000,0.0000,0.0000,0.000,0.000
S2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00,0.000,0.3838,0.0000,0.0000,0.00,0.0000,0.0000,0.3838,0.0000,0.000,0.00,0.0000,0.0000,0.0000,0.000,0.000,0.0000,0.0000,0.0000,0.2912,0.0000,0.0000,0.0000,0.0000,0.00,0.0000,0.0000,0.0000,0.3296,0.3838,0.000,0.3296,0.0000,0.0000,0.0000,0.0000,0.3838,0.0000,0.0000,0.000,0.0000,0.00,0.0000,0.00,0.0000,0.0000,0.0000,0.0000,0.0000,0.3296,0.0000,0.000,0.000
S3,0.0000,0.0000,0.0000,0.4338,0.0000,0.0000,0.00,0.000,0.0000,0.0000,0.0000,0.00,0.0000,0.3291,0.0000,0.0000,0.000,0.00,0.0000,0.0000,0.0000,0.000,0.000,0.4338,0.0000,0.0000,0.0000,0.0000,0.4338,0.0000,0.0000,0.00,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.4338,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.00,0.3726,0.00,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.000
S4,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00,0.000,0.0000,0.0000,0.3205,0.00,0.0000,0.0000,0.0000,0.0000,0.000,0.00,0.2752,0.3205,0.0000,0.000,0.000,0.0000,0.6409,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00,0.0000,0.3205,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.3205,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.00,0.0000,0.00,0.0000,0.0000,0.3205,0.0000,0.0000,0.0000,0.0000,0.000,0.000
S5,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00,0.393,0.0000,0.3375,0.0000,0.00,0.0000,0.0000,0.0000,0.0000,0.000,0.00,0.0000,0.0000,0.0000,0.393,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3375,0.00,0.0000,0.0000,0.0000,0.0000,0.0000,0.393,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.00,0.0000,0.00,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.393,0.393
S6,0.0000,0.0000,0.3688,0.0000,0.0000,0.0000,0.00,0.000,0.0000,0.0000,0.0000,0.00,0.0000,0.5595,0.0000,0.0000,0.000,0.00,0.0000,0.0000,0.0000,0.000,0.000,0.0000,0.0000,0.0000,0.2798,0.3167,0.0000,0.0000,0.0000,0.00,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3688,0.0000,0.000,0.3688,0.00,0.0000,0.00,0.0000,0.0000,0.0000,0.0000,0.0000,0.3167,0.0000,0.000,0.000
S7,0.4037,0.0000,0.0000,0.0000,0.0000,0.0000,0.00,0.000,0.0000,0.0000,0.0000,0.00,0.4701,0.0000,0.0000,0.0000,0.000,0.00,0.0000,0.0000,0.0000,0.000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.4037,0.0000,0.0000,0.3566,0.000,0.0000,0.00,0.4037,0.00,0.0000,0.4037,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.000
S8,0.0000,0.0000,0.0000,0.0000,0.4649,0.0000,0.00,0.000,0.0000,0.3993,0.0000,0.00,0.0000,0.0000,0.0000,0.0000,0.000,0.00,0.0000,0.0000,0.0000,0.000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00,0.0000,0.0000,0.3527,0.3993,0.0000,0.000,0.0000,0.0000,0.4649,0.0000,0.0000,0.0000,0.0000,0.3527,0.000,0.0000,0.00,0.0000,0.00,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.000
S9,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00,0.000,0.0000,0.0000,0.0000,0.00,0.0000,0.0000,0.0000,0.0000,0.438,0.00,0.0000,0.0000,0.0000,0.000,0.438,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.00,0.0000,0.0000,0.0000,0.0000,0.0000,0.00

In [48]:
# ------------------------------------------------------------
# Q4(d) Cosine similarity for at least five sentence pairs
# ------------------------------------------------------------

q4_pairs = [
    (1, 2),
    (1, 6),
    (2, 5),
    (3, 4),
    (7, 8),
    (9, 10),
    (11, 12),
    (1, 7)
]

q4_pair_results = []

print("Sentence Cosine Similarities:")
print("=" * 60)

for a, b in q4_pairs:
    score = cosine_similarity(
        q4_matrix[a - 1],
        q4_matrix[b - 1]
    )[0, 0]

    q4_pair_results.append([f"S{a}", f"S{b}", score])
    print(f"S{a} vs S{b}: {score:.4f}")

q4_pair_df = pd.DataFrame(
    q4_pair_results,
    columns=["Sentence 1", "Sentence 2", "Cosine Similarity"]
)

display(q4_pair_df.round(4))


Sentence Cosine Similarities:
S1 vs S2: 0.1998
S1 vs S6: 0.3602
S2 vs S5: 0.0000
S3 vs S4: 0.0000
S7 vs S8: 0.1258
S9 vs S10: 0.0000
S11 vs S12: 0.0000
S1 vs S7: 0.0000


,Sentence 1,Sentence 2,Cosine Similarity
0,S1,S2,0.1998
1,S1,S6,0.3602
2,S2,S5,0.0000
3,S3,S4,0.0000
4,S7,S8,0.1258
5,S9,S10,0.0000
6,S11,S12,0.0000
7,S1,S7,0.0000


In [49]:
# ------------------------------------------------------------
# Q4(e) Identify sentences with similar content
# ------------------------------------------------------------

q4_similarity = cosine_similarity(q4_matrix)

all_sentence_pairs = []

for i in range(len(q4_sentences)):
    for j in range(i + 1, len(q4_sentences)):
        all_sentence_pairs.append(
            (
                q4_similarity[i, j],
                i + 1,
                j + 1
            )
        )

all_sentence_pairs.sort(reverse=True)

print("Top 5 most similar sentence pairs:")
print("=" * 80)

for score, a, b in all_sentence_pairs[:5]:
    print(f"S{a} vs S{b}: {score:.4f}")
    print(f"S{a}: {q4_sentences[a - 1]}")
    print(f"S{b}: {q4_sentences[b - 1]}")
    print()


Top 5 most similar sentence pairs:
S1 vs S6: 0.3602
S1: Machine learning models learn useful patterns from training data.
S6: Data science uses statistics and machine learning to analyze data.

S7 vs S9: 0.3037
S7: Cyber security protects systems from unauthorized access.
S9: Encryption protects sensitive information from unauthorized users.

S8 vs S12: 0.2205
S8: Network security prevents attacks on computer networks.
S12: Security monitoring helps detect malicious network activity.

S1 vs S2: 0.1998
S1: Machine learning models learn useful patterns from training data.
S2: Deep learning uses neural networks to recognize complex patterns.

S2 vs S6: 0.1859
S2: Deep learning uses neural networks to recognize complex patterns.
S6: Data science uses statistics and machine learning to analyze data.



In [50]:
# ------------------------------------------------------------
# Q4(f) Combine sentence representations into document vectors
# ------------------------------------------------------------

# First 6 sentences form the AI document.
# Last 6 sentences form the Security document.

ai_document_vector = np.asarray(
    q4_matrix[:6].mean(axis=0)
).flatten()

security_document_vector = np.asarray(
    q4_matrix[6:].mean(axis=0)
).flatten()

document_vectors = pd.DataFrame(
    [ai_document_vector, security_document_vector],
    index=["AI_Document", "Security_Document"],
    columns=q4_feature_names
)

print("Document-to-vector matrix dimensions:", document_vectors.shape)
display(document_vectors.round(4))

document_similarity = cosine_similarity(
    ai_document_vector.reshape(1, -1),
    security_document_vector.reshape(1, -1)
)[0, 0]

print(
    f"Cosine similarity between AI and Security documents: "
    f"{document_similarity:.4f}"
)


Document-to-vector matrix dimensions: (2, 60)


,access,activity,analyze,artificial,attacks,authentication,block,classify,complex,computer,computers,connections,cyber,data,deep,detect,encryption,firewalls,helps,human,identity,images,information,intelligence,language,learn,learning,machine,make,malicious,models,monitor,monitoring,natural,network,networks,neural,objects,patterns,predictions,prevents,processing,protects,recognize,science,security,sensitive,statistics,suspicious,systems,traffic,training,unauthorized,understand,useful,users,uses,verifies,videos,vision
AI_Document,0.0000,0.0000,0.0615,0.0723,0.0000,0.0000,0.000,0.0655,0.064,0.0563,0.0534,0.000,0.0000,0.1982,0.064,0.0000,0.000,0.000,0.0459,0.0534,0.0000,0.0655,0.000,0.0723,0.1068,0.0661,0.1453,0.1095,0.0723,0.0000,0.113,0.000,0.0000,0.0534,0.0000,0.0549,0.064,0.0655,0.1117,0.0723,0.0000,0.0534,0.00,0.064,0.0615,0.0000,0.000,0.0615,0.000,0.0621,0.000,0.0661,0.00,0.0534,0.0661,0.0000,0.1077,0.0000,0.0655,0.0655
Security_Document,0.1349,0.0687,0.0000,0.0000,0.0775,0.0788,0.065,0.0000,0.000,0.0665,0.0000,0.065,0.0783,0.0000,0.000,0.0687,0.073,0.065,0.0590,0.0000,0.0788,0.0000,0.073,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0687,0.000,0.065,0.0687,0.0000,0.1602,0.0665,0.000,0.0000,0.0000,0.0000,0.0775,0.0000,0.13,0.000,0.0000,0.1703,0.073,0.0000,0.065,0.0673,0.065,0.0000,0.13,0.0000,0.0000,0.1304,0.0000,0.0788,0.0000,0.0000


Cosine similarity between AI and Security documents: 0.0611


In [51]:
# ------------------------------------------------------------
# Q4(g) Compare word, sentence and document representations
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Representation": [
        "Word vector (Q3 co-occurrence)",
        "Sentence vector (Q4 TF-IDF)",
        "Document vector (Q4 average sentence vectors)"
    ],

    "Number of units represented": [
        len(selected_words),
        len(q4_sentences),
        2
    ],

    "Vector dimension": [
        len(context_words),
        q4_matrix.shape[1],
        len(q4_feature_names)
    ],

    "Meaning of each dimension": [
        "Context-word co-occurrence",
        "TF-IDF vocabulary term",
        "TF-IDF vocabulary term aggregated across sentences"
    ]
})

print("Comparison of Representations:")
display(comparison)

print("\nKey comparison:")
print("- Word vectors describe distributional context around words.")
print("- Sentence vectors represent individual sentences.")
print("- Document vectors summarize multiple sentence vectors into one fixed-length vector.")


Comparison of Representations:


,Representation,Number of units represented,Vector dimension,Meaning of each dimension
0,Word vector (Q3 co-occurrence),15,118,Context-word co-occurrence
1,Sentence vector (Q4 TF-IDF),12,60,TF-IDF vocabulary term
2,Document vector (Q4 average sentence vectors),2,60,TF-IDF vocabulary term aggregated across sentences



Key comparison:
- Word vectors describe distributional context around words.
- Sentence vectors represent individual sentences.
- Document vectors summarize multiple sentence vectors into one fixed-length vector.
